# 1. 모듈 불러오기 및 기본 설정

In [49]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy as stats
import seaborn as sns
import numpy as np
import platform

# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False

# 출력 짤림 방지
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# 2.원본 데이터 불러오기 및 결측치 탐색

In [50]:
# 데이터 불러오기
df = pd.read_csv('./data/Courses.csv', parse_dates=['start_time_DI', 'last_event_DI'])
df.head()


,index,course_id,userid_DI,registered,viewed,explored,certified,final_cc_cname_DI,LoE_DI,YoB,gender,grade,start_time_DI,last_event_DI,nevents,ndays_act,nplay_video,nchapters,nforum_posts,roles,incomplete_flag
0,0,HarvardX/CB22x/2013_Spring,MHxPC130442623,1,0,0,0,United States,NaN,NaN,NaN,0,2012-12-19,2013-11-17,NaN,9.0,NaN,NaN,0,NaN,1.0
1,1,HarvardX/CS50x/2012,MHxPC130442623,1,1,0,0,United States,NaN,NaN,NaN,0,2012-10-15,NaT,NaN,9.0,NaN,1.0,0,NaN,1.0
2,2,HarvardX/CB22x/2013_Spring,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2013-02-08,2013-11-17,NaN,16.0,NaN,NaN,0,NaN,1.0
3,3,HarvardX/CS50x/2012,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2012-09-17,NaT,NaN,16.0,NaN,NaN,0,NaN,1.0
4,4,HarvardX/ER22x/2013_Spring,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2012-12-19,NaT,NaN,16.0,NaN,NaN,0,NaN,1.0


In [51]:
# 원본데이터 결측치 개수, 비율
display(pd.DataFrame({
    'sum': df.isna().sum(),
    'ratio': df.isna().mean() * 100
}).sort_values('ratio', ascending=False).reset_index())

,index,sum,ratio
0,roles,641138,100.000000
1,incomplete_flag,540977,84.377622
2,nplay_video,457530,71.362172
3,nchapters,258753,40.358394
4,nevents,199151,31.062111
5,last_event_DI,178954,27.911932
6,ndays_act,162743,25.383459
7,LoE_DI,106008,16.534350
8,YoB,96605,15.067739
9,gender,86806,13.539363


# 전처리 시작
- 전처리용 데이터 셋 생성
- 컬럼명 소문자로 변환
- 중복행 확인
- 컬럼 1차 제거
    - >index: 인덱스 컬럼
    - >roles: 모든 행이 결측으로 이루어진 컬럼
- 데이터 형변환
- 파생컬럼 생성
    -  `age`: 나이
        -  `age_segment`: 연령대 (`age`컬럼 활용)
    -  `step`: 사용자별 퍼널 단계
    -  `missing_flag` : 결측 패턴 유지를 위한 플래그
        -  `nchapters_flag` : nchapter의 결측 패턴을 저장
        -  `ndays_act_flag` : 
        -  `nplay_video_flag` :
        -  `last_event_di_flag` :
        -  `age_flag` :
        -  `grade_flag` :
    - `duration` : 강좌 내 활동 기간
- 행 제거
    - 논리 오류 위주의 행 제거작업 수행
- 결측치 대체
    - `gender`, `LoE` : `unknown`으로 대체
    - ``

In [52]:
# 전처리용 데이터 셋 생성
pre = df.copy()

In [53]:
# 컬럼명 소문자로 변환
pre.columns = pre.columns.str.lower()

In [54]:
# 중복행 확인
pre.duplicated().sum()

np.int64(0)

In [55]:
# 의미없는 컬럼 제거
pre = pre.drop(columns=['index', 'roles'])

In [56]:
#데이터 형변환
# grade 숫자형으로 변환
pre['grade'] = pd.to_numeric(pre['grade'], errors='coerce')

In [57]:
# 파생컬럼 생성

# 학생들의 나이(age) & (age_segment)
pre['age'] = pre['start_time_di'].dt.year - pre['yob']

pre['age_segment'] = np.select(
    [
        pre['age'].isna(),
        pre['age'] >= 60,
        pre['age'] >= 50,
        pre['age'] >= 40,
        pre['age'] >= 30,
        pre['age'] >= 20,
    ],
    [
        'unknown',
        '60s+',
        '50s',
        '40s',
        '30s',
        '20s',
    ],
    default='under 20'
)

# 퍼널 단계 컬럼(step): 각 학생 별 진행 단계
pre['step'] = np.select(
    [
        pre['certified'] ==1,
        pre['explored'] == 1,
        pre['viewed'] == 1,
        pre['registered'] == 1,
    ],
    [
        'c',
        'e',
        'v',
        'r'
    ],
    default='None'
)

# Missing Flag 컬럼 생성
missing_col = [
    'nchapters', 
    'nevents', 
    'ndays_act', 
    'nplay_video', 
    'last_event_di', 
    'age', 
    'grade']

for col in missing_col:
    pre[f'{col}_flag'] = pre[col].isna().astype(int)
    

# 학습 기간 (duration) 컬럼 생성 (같은 날일 경우 1)
pre['duration'] = ((pre['last_event_di'] - pre['start_time_di'])).dt.days.astype(int, errors='ignore') + 1

In [58]:
a = df.loc[
    (df['registered']== 1) &
    (df['viewed'] == 1) &
    (df['explored']==0) &
    (df['certified']==0) &
    (df['nplay_video'].notna()),
    'nplay_video']


b = df.loc[
    (df['registered']== 1) &
    (df['viewed'] == 0) &
    (df['explored']==0) &
    (df['certified']==0) &
    (df['nplay_video'].notna()),
    'nplay_video']



In [59]:
# 행제거
print('행 제거 작업 시작 전:')
print(pre.shape)

# 퍼널 논리적 오류 행 제거
funnel_mask1 = (pre['viewed'] == 0) & (pre['explored'] == 1)
funnel_mask2 = (pre['explored'] == 0) & (pre['certified'] == 1)
pre = pre[~funnel_mask1]
pre = pre[~funnel_mask2]

# durration 음수 행 제거
duration_mask = pre['duration'] <= 0
pre = pre[~duration_mask]

# age 13세 미만 행 제거
age_mask = pre['age'] < 13
pre = pre[~age_mask]

# 상시 개방된 강의 제거 
course_mask = (pre['course_id'] =='HarvardX/CS50x/2012') | (pre['course_id'] =='HarvardX/ER22x/2013_Spring') | (pre['course_id'] =='HarvardX/CB22x/2013_Spring')
pre = pre[~course_mask]

# incomplete_flag == 1 제외
pre = pre[pre['incomplete_flag'].isna()]

# 논리적 오류 drop 
rchap_mask = (pre['step']=='r') & ((pre['nchapters'] > 0) | (pre['nplay_video']>0))
pre = pre[~rchap_mask]

# 행동지표가 모두 nan인 행 제외
nan_mask = (pre['nevents'].isna()) & (pre['ndays_act'].isna()) & (pre['last_event_di'].isna())
pre = pre[~nan_mask]

print('행 제거 작업 후:')
print(pre.shape)

행 제거 작업 시작 전:
(641138, 30)


C:\Users\gmltk\AppData\Local\Temp\ipykernel_40352\1718670232.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  pre = pre[~funnel_mask2]


행 제거 작업 후:
(324228, 30)


In [60]:
# 결측치 대체

# 성별 결측치(gender) : unknown 대체
pre['gender'] = pre['gender'].fillna('unknown')

# 학력 결측치(LoE_DI) : unknown 대체
pre['loe_di'] = pre['loe_di'].fillna('unknown')

# 탐색한 챕터 수 결측치(nchapters) : registered 단계일 때 0으로 대체
pre.loc[
    (pre['step']=='r') & (pre['nchapters'].isna()),
    'nchapters'
] = 0

# 탐색한 챕터 수 결측치(nchapters) : viewed 단계일 때 2로 대체
v_nchapters_median = pre.loc[
    (pre['step']=='v') & (pre['nchapters'].notna()),
    'nchapters'
].median()

pre.loc[
    (pre['step']=='v') & (pre['nchapters'].isna()),
    'nchapters'
] = v_nchapters_median

# 총 이벤트 발생 수 결측치(nevent) : registered 단계일 때 0으로 대체
# pre.loc[
#     (pre['step']=='r') & (pre['nevents'].isna()),
#     'nevents'
# ] = 0

# # 활성 일수 결측치(ndays_act) : 0으로 대체
# pre['ndays_act'] = pre['ndays_act'].fillna(0)

# 영상재생횟수(nplay_video) 결측치
pre.loc[
    (pre['nchapters']==0) & (pre['nplay_video'].isna()),
    'nplay_video'
] = 0

# 영상재생횟수 Viewed 단계 결측치 중앙값으로 대체
v_play_median = pre.loc[
    (pre['step']=='v') & (pre['nplay_video'].notna()),
    'nplay_video'
].median()

pre.loc[
    (pre['step']=='v') & (pre['nplay_video'].isna()),
    'nplay_video'
] = v_play_median

# 영상재생횟수 Explored 단계 결측치 중앙값으로 대체
e_play_median = pre.loc[
    (pre['step']=='e') & (pre['nplay_video'].notna()),
    'nplay_video'
].median()

pre.loc[
    (pre['step']=='e') & (pre['nplay_video'].isna()),
    'nplay_video'
] = e_play_median

# 영상재생횟수 Certified 단계 결측치 중앙값으로 대체
c_play_median = pre.loc[
    (pre['step']=='c') & (pre['nplay_video'].notna()),
    'nplay_video'
].median()

pre.loc[
    (pre['step']=='c') & (pre['nplay_video'].isna()),
    'nplay_video'
] = c_play_median



# 마지막 이벤트 발생일 (last_event_DI) : 논의중

# 나이 (age) 결측치 : 논의중

# 성적(grade) 결측치 : 논의중

# 강좌내 활동 기간 (duration)
pre['duration'] = pre['duration'].fillna(0)


In [61]:
pre.isna().sum()

course_id                  0
userid_di                  0
registered                 0
viewed                     0
explored                   0
certified                  0
final_cc_cname_di          0
loe_di                     0
yob                    55542
gender                     0
grade                  31609
start_time_di              0
last_event_di              0
nevents                    0
ndays_act                  0
nplay_video                0
nchapters                  0
nforum_posts               0
incomplete_flag       324228
age                    55542
age_segment                0
step                       0
nchapters_flag             0
nevents_flag               0
ndays_act_flag             0
nplay_video_flag           0
last_event_di_flag         0
age_flag                   0
grade_flag                 0
duration                   0
dtype: int64

In [62]:
# 강의별 공식 일정 테이블 생성 & merge

# 1. course_id별 공식 일정 매핑표
schedule_map = {
        # HarvardX
        'HarvardX/PH207x/2012_Fall': {
                'course_title': 'Health in Numbers: Quantitative Methods in Clinical & Public Health Research',
                'registration_open': '2012-07-24',
                'course_launch': '2012-10-15',
                'course_wrap': '2013-01-30'
        },
        'HarvardX/CS50x/2012': {
                'course_title': 'Introduction to Computer Science I',
                'registration_open': '2012-07-24',
                'course_launch': '2012-10-15',
                        'course_wrap': '2013-04-15'
                },
                'HarvardX/HLS1x/2013_Spring': {
                        'course_title': 'Copyright',
                        'registration_open': '2012-12-19',
                        'course_launch': '2013-01-28',
                        'course_wrap': '2013-07-03'
                },
                'HarvardX/ER22x/2013_Spring': {
                        'course_title': 'Justice',
                        'registration_open': '2012-12-19',
                        'course_launch': '2013-03-02',
                        'course_wrap': '2013-07-26'
                },
                'HarvardX/CB22x/2013_Spring': {
                        'course_title': 'The Ancient Greek Hero',
                        'registration_open': '2012-12-19',
                        'course_launch': '2013-03-13',
                        'course_wrap': '2013-08-26'
                },
                'HarvardX/PH278x/2013_Spring': {
                        'course_title': 'Human Health and Global Environmental Change',
                        'registration_open': '2012-12-19',
                        'course_launch': '2013-05-15',
                        'course_wrap': '2013-07-25'
                },

                # MITx
                'MITx/6.002x/2012_Fall': {
                        'course_title': 'Circuits and Electronics - Fall',
                        'registration_open': '2012-07-24',
                        'course_launch': '2012-09-05',
                        'course_wrap': '2012-12-25'
                },
                'MITx/6.00x/2012_Fall': {
                        'course_title': 'Introduction to Computer Science and Programming - Fall',
                        'registration_open': '2012-07-24',
                        'course_launch': '2012-09-26',
                        'course_wrap': '2013-01-15'
                },
                'MITx/3.091x/2012_Fall': {
                        'course_title': 'Introduction to Solid State Chemistry - Fall',
                        'registration_open': '2012-07-24',
                        'course_launch': '2012-10-09',
                        'course_wrap': '2013-01-15'
                },
                'MITx/6.00x/2013_Spring': {
                        'course_title': 'Introduction to Computer Science and Programming - Spring',
                        'registration_open': '2012-12-19',
                        'course_launch': '2013-02-04',
                        'course_wrap': '2013-06-04'
                },
                'MITx/3.091x/2013_Spring': {
                        'course_title': 'Introduction to Solid State Chemistry - Spring',
                        'registration_open': '2012-12-20',
                        'course_launch': '2013-02-05',
                        'course_wrap': '2013-06-21'
                },
                'MITx/14.73x/2013_Spring': {
                        'course_title': 'The Challenges of Global Poverty',
                        'registration_open': '2012-12-19',
                        'course_launch': '2013-02-12',
                        'course_wrap': '2013-05-21'
                },
                'MITx/8.02x/2013_Spring': {
                        'course_title': 'Electricity and Magnetism',
                        'registration_open': '2013-01-17',
                        'course_launch': '2013-02-18',
                        'course_wrap': '2013-06-18'
                },
                'MITx/6.002x/2013_Spring': {
                        'course_title': 'Circuits and Electronics - Spring',
                        'registration_open': '2012-12-20',
                        'course_launch': '2013-03-03',
                        'course_wrap': '2013-07-01'
                },
                'MITx/7.00x/2013_Spring': {
                        'course_title': 'Introduction to Biology - The Secret of Life',
                        'registration_open': '2013-01-30',
                        'course_launch': '2013-03-05',
                        'course_wrap': '2013-06-06'
                },
                'MITx/2.01x/2013_Spring': {
                        'course_title': 'Elements of Structures',
                        'registration_open': '2013-02-24',
                        'course_launch': '2013-04-15',
                        'course_wrap': '2013-07-30'
                },
                'MITx/8.MReV/2013_Summer': {
                        'course_title': 'Mechanics ReView',
                        'registration_open': '2013-04-27',
                        'course_launch': '2013-06-01',
                        'course_wrap': '2013-09-15'
                }
}

# 2. dict -> DataFrame 변환
schedule_df = (
        pd.DataFrame(schedule_map)
        .T
        .reset_index()
        .rename(columns={'index': 'course_id'})
)

# 3. 날짜형 변환
date_cols = ['registration_open', 'course_launch', 'course_wrap']
for col in date_cols:
        schedule_df[col] = pd.to_datetime(schedule_df[col])

# pre = pd.merge()

In [63]:
# 2차 컬럼 drop
pre = pre.drop(columns=['yob', 'incomplete_flag'])

In [64]:
pre[pre['viewed']==1].isna().sum().sort_values(ascending=False)

age                   43054
grade                 21246
registered                0
course_id                 0
viewed                    0
explored                  0
final_cc_cname_di         0
certified                 0
loe_di                    0
gender                    0
start_time_di             0
userid_di                 0
last_event_di             0
nevents                   0
nplay_video               0
ndays_act                 0
nchapters                 0
nforum_posts              0
age_segment               0
step                      0
nchapters_flag            0
nevents_flag              0
ndays_act_flag            0
nplay_video_flag          0
last_event_di_flag        0
age_flag                  0
grade_flag                0
duration                  0
dtype: int64